# --- Day 12: Hill Climbing Algorithm ---

You try contacting the Elves using your handheld device, but the river you're following must be too low to get a decent signal.

You ask the device for a heightmap of the surrounding area (your puzzle input). The heightmap shows the local area from above broken into a grid; the elevation of each square of the grid is given by a single lowercase letter, where a is the lowest elevation, b is the next-lowest, and so on up to the highest elevation, z.

Also included on the heightmap are marks for your current position (S) and the location that should get the best signal (E). Your current position (S) has elevation a, and the location that should get the best signal (E) has elevation z.

You'd like to reach E, but to save energy, you should do it in as few steps as possible. During each step, you can move exactly one square up, down, left, or right. To avoid needing to get out your climbing gear, the elevation of the destination square can be at most one higher than the elevation of your current square; that is, if your current elevation is m, you could step to elevation n, but not to elevation o. (This also means that the elevation of the destination square can be much lower than the elevation of your current square.)

For example:
```
Sabqponm
abcryxxl
accszExk
acctuvwj
abdefghi
```
Here, you start in the top-left corner; your goal is near the middle. You could start by moving down or right, but eventually you'll need to head toward the e at the bottom. From there, you can spiral around to the goal:
```
v..v<<<<
>v.vv<<^
.>vv>E^^
..v>>>^^
..>>>>>^
```
In the above diagram, the symbols indicate whether the path exits each square moving up (^), down (v), left (<), or right (>). The location that should get the best signal is still E, and . marks unvisited squares.

This path reaches the goal in 31 steps, the fewest possible.

What is the fewest steps required to move from your current position to the location that should get the best signal?


## Read input file

In [1]:
with open('input.txt', 'r') as file:
    heightmap = [[*line.strip()] for line in file.read().strip().split('\n')]

## Solution

In [2]:
from queue import PriorityQueue

heights = {'S': 'a', 'E': 'z'}

def find(heightmap, symbol):
    for i, row in enumerate(heightmap):
        for j, cell in enumerate(row):
            if cell == symbol:
                return (i, j)

            
def list_reachables(coords, visited, max_coords, can_reach):
    nexts = [
        (coords[0]-1, coords[1]),
        (coords[0]+1, coords[1]),
        (coords[0], coords[1]-1),
        (coords[0], coords[1]+1)
    ]


    nexts = [c for c in nexts if c[0] >= 0 and c[1] >= 0]
    nexts = [c for c in nexts if c[0] < max_coords[0] and c[1] < max_coords[1]]
    nexts = [c for c in nexts if c not in visited]
    nexts = [c for c in nexts if can_reach(coords, c)]
    return nexts

def reachable(from_coords, to_coords):
    val = lambda x : ord(heights.get(x, x))
    return val(heightmap[from_coords[0]][from_coords[1]]) >= (val(heightmap[to_coords[0]][to_coords[1]]) - 1)

queue = PriorityQueue()
visited = set()
cost = dict()

start = find(heightmap, 'S')
end = find(heightmap, 'E')

cost[start] = 0

queue.put((0, start))
while not queue.empty():
    pos = queue.get()

    if pos[1] in visited:
        continue
    visited.add(pos[1])
    cost[pos[1]] = pos[0]
    if pos[1] == end:
        break

    for p in list_reachables(pos[1], visited, (len(heightmap), len(heightmap[0])), reachable):
        queue.put((cost[pos[1]]+1, p))
display(f"Steps: {cost[end]}")

'Steps: 380'

# --- Part Two ---

As you walk up the hill, you suspect that the Elves will want to turn this into a hiking trail. The beginning isn't very scenic, though; perhaps you can find a better starting point.

To maximize exercise while hiking, the trail should start as low as possible: elevation a. The goal is still the square marked E. However, the trail should still be direct, taking the fewest steps to reach its goal. So, you'll need to find the shortest path from any square at elevation a to the square marked E.

Again consider the example from above:
```
Sabqponm
abcryxxl
accszExk
acctuvwj
abdefghi
```
Now, there are six choices for starting position (five marked a, plus the square marked S that counts as being at elevation a). If you start at the bottom-left square, you can reach the goal most quickly:
```
...v<<<<
...vv<<^
...v>E^^
.>v>>>^^
>^>>>>>^
```
This path reaches the goal in only 29 steps, the fewest possible.

What is the fewest steps required to move starting from any square with elevation a to the location that should get the best signal?


## Solution

In [3]:
def find_all(heightmap, symbol):
    coords = list()
    for i, row in enumerate(heightmap):
        for j, cell in enumerate(row):
            if cell == symbol:
                coords.append((i, j))
    return coords

queue = PriorityQueue()
visited = set()
cost = dict()

start = find(heightmap, 'S')
end = find(heightmap, 'E')

cost[end] = 0

queue.put((0, end))
while not queue.empty():
    pos = queue.get()

    if pos[1] in visited:
        continue
    visited.add(pos[1])
    cost[pos[1]] = pos[0]

    for p in list_reachables(pos[1], visited, (len(heightmap), len(heightmap[0])), lambda a,b : reachable(b,a)):
        queue.put((cost[pos[1]]+1, p))


min_steps = cost[start]
for p in find_all(heightmap, 'a'):
    if p in visited:
        min_steps = min(min_steps, cost[p])
min_steps

375